# 04 · Return R3: antenatal prediction, population and count audit — Fetal death, MIMIC-IV (M3 · R3)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 · G1 → 02: S4 → 03: S5 · S6 → R3 → **04: R3 · S2 · S3 · G1 · S4** → 05: S5 · S6 · S7 · G2 → 06: G3 · class · S11 |
| What it does | Applies return R3 (another prediction point): with what was known of the woman **before** the delivery admission, can a high risk of fetal death be seen? It builds or checks the obstetric universe and the label of this prediction point (S2, S3), reproduces the reference counts (G1) and audits how much information exists before the admission (S4) |
| Stages and gates | R3 · S2 · S3 · G1 · S4 |
| Decisions | Prediction point at admission with information recorded at least 7 days before; predictors fixed with the frequencies of the whole population; see `docs/outcome_card_antenatal.md` |
| Inputs | MIMIC-IV hosp tables (`admissions`, `patients`, `diagnoses_icd`, `procedures_icd`, `omr`, `labevents`, `d_labitems`) |
| Tables of the mimiciv schema | `antenatal_obstetric_universe` (`sql/03_antenatal/01_obstetric_universe.sql`) and `antenatal_fetal_death_label` (`sql/03_antenatal/02_fetal_death_label.sql`) |
| Script | `sql/03_antenatal/03_prior_information.sql` (read-only) |
| Outputs | `data/outputs_04_antenatal_count_audit.txt` (aggregate counts only; 1 to 4 shown as "<5") |
| Time | Several minutes: the audit reads `labevents` for the patients of the population |
| Previous / next | `03_unit_and_prediction_point` / `05_antenatal_predictors` |
| Spanish version | `04_antenatal_count_audit_es.ipynb` (same code) |

## R3 · Return: another prediction point

Notebook 03 found that more than half of the fetal deaths were already present at admission + 6 h, so a model at that point was not fitted. Return R3 moves the question to a moment when the death had not happened yet. **Prediction point:** the admission time, using only information recorded **at least 7 days before** (earlier admissions, outpatient measurements and outpatient laboratory values); 1 and 30 days as sensitivity. The 7-day margin leaves out what was recorded after a death diagnosed in an outpatient visit, which is usually followed by an admission within a few days. The outcome, the population and the unit do not change; the universe and the label are rebuilt from the MIMIC-IV tables, so that this prediction point does not depend on any table of the earlier one.

| Result | What is done | In this model |
|---|---|---|
| The model at the first prediction point cannot be fitted (C1 anticipated) and a return is still available | R3: move the prediction point before the admission | 1 of the 2 returns allowed |

> **Rebuild the tables.** The tables of the `mimiciv` schema that this notebook needs are read as they are, and only their checks run. If a table does not exist, it is built from its SQL script. To rebuild it (for example after a change in a script of `sql/`), set **`REBUILD = True`** in the first code cell and run the notebook again.

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_04_antenatal_count_audit.txt"
engine = get_connect_database()

## S2 · Universe and population of the antenatal prediction point

The obstetric universe (admissions of women aged 12 to 55 with a diagnosis of pregnancy, delivery or puerperium) and the delivery flag are rebuilt here from the MIMIC-IV tables. The population is the same as in notebook 01: delivery hospitalizations plus the admissions with a stillbirth code outside the delivery flag.

| Result | What is done | In this model |
|---|---|---|
| The counts reproduce notebook 01 | On to S3 | 25,736 admissions; population 19,440 |
| They differ | Review `sql/03_antenatal/01_obstetric_universe.sql` | — |

In [ ]:
# Stage S2: obstetric universe of the antenatal prediction point: mimiciv.antenatal_obstetric_universe
resu = ensure_table(engine, "antenatal_obstetric_universe", SQL / "03_antenatal" / "01_obstetric_universe.sql", REBUILD, log=say)
show_results(resu, ["Q-U1 reference counts", "Q-U2 population by evidence and era"])

## S3 · Outcome definition — the same label, rebuilt for this prediction point

Strict definition (fetal death of the whole pregnancy: ICD-10-CM O36.4, Z37.1, Z37.4, Z37.7; ICD-9-CM 656.4, V27.1, V27.4, V27.7) as the main label; the other definitions as sensitivity, as in decision D5.

| Result | What is done | In this model |
|---|---|---|
| The catalogue has the 18 patterns and the label sizes match notebook 01 | On to G1 | 129 strict events; 142 with partial codes |
| They differ | Review `sql/03_antenatal/02_fetal_death_label.sql` | — |

In [ ]:
# Stage S3: label by admission of the antenatal prediction point: mimiciv.antenatal_fetal_death_label
resl = ensure_table(engine, "antenatal_fetal_death_label", SQL / "03_antenatal" / "02_fetal_death_label.sql", REBUILD, log=say)
show_results(resl, ["Q-L1 code patterns by tier", "Q-L3 label sizes"])

## G1 · Definition gate — the same population and events as notebook 01

The reference counts are those of notebook 01 and of the count audit. If any differs, the notebook stops.

| Result | What is done | In this model |
|---|---|---|
| Every count is reproduced | G1 passes: on to the audit of the information before admission (S4) | Passes |
| A count differs | It stops (`assert`): go back to S2 or S3 | — |

In [ ]:
# Gate G1: reference counts (notebook 01 and the count audit)
u1, l2 = resu["Q-U1 reference counts"].iloc[0], resl["Q-L2 reference counts"].iloc[0]
l3 = resl["Q-L3 label sizes"].iloc[0]
g1_ok = check([
    ("obstetric admissions", int(u1.n_obstetric_admissions), 25736),
    ("patients of the universe", int(u1.n_patients), 15084),
    ("delivery hospitalizations", int(u1.n_delivery_hospitalizations), 19425),
    ("population (delivery flag or stillbirth code)", int(u1.n_population), 19440),
    ("population: admissions", int(l2.n_admissions), 19440),
    ("population: patients", int(l2.n_patients), 13735),
    ("strict events (main label)", int(l2.n_strict), 129),
    ("events with partial codes", int(l2.n_events_any), 142),
    ("strict events without an early-loss code", int(l3.n_strict_without_early_loss_code), 119),
    ("strict events with 22 weeks or more", int(l3.n_ge22_weeks), 55),
    ("strict events with 28 weeks or more", int(l3.n_ge28_weeks), 37),
], "G1 · Reference counts")
decision("G1 · Definition gate", [
    ("population, unit and outcome of notebook 01, rebuilt", "passes" if g1_ok else "DOES NOT PASS",
     "on to S4" if g1_ok else "stop: go back to S2 or S3")])
assert g1_ok, "gate G1 does not pass: a reference count differs"

## S4 · Count audit — how much is known before the admission? (feasibility)

Three questions before any model: how many women have some information before the admission (if almost none, the model is not feasible); what information there is, how often and how recent; and, with that, which predictors enter. **Rule against selection by outcome:** only the availability of whole blocks (Q-A1 to Q-A3) is split by events and non-events, because it decides feasibility; the frequency of each condition or measurement (Q-A4 to Q-A7) is shown for the whole population only.

| Result | What is done | In this model |
|---|---|---|
| Most events have some information before the admission | The model is feasible: on to the predictors (notebook 05) | 99 of 129 events (76.7%) with the 7-day margin |
| Almost no event has information | Not feasible: next rung of the outcome ladder (R1) or another return | — |
| Rare conditions (less than 1% of the population) | Joined in composites, never chosen by their association with the outcome | Two composites: adverse outcome in an earlier pregnancy; other high-risk chronic condition |

In [ ]:
# Stage S4: information available before the admission (read-only script; several minutes)
pri = run_script(engine, SQL / "03_antenatal" / "03_prior_information.sql", log=say)
show_results(pri, ["Q-A0 population check", "Q-A1 information available before admission", "Q-A2 information by period (7 days)",
                   "Q-A3 earlier admissions (7 days)", "Q-A4 conditions in earlier admissions (7 days)",
                   "Q-A5 outpatient measurements (7 days)", "Q-A6 outpatient laboratory values (7 days)",
                   "Q-A7 registration variables"])
qa1 = pri["Q-A1 information available before admission"]
ev7 = qa1[(qa1.buffer_days == 7) & (qa1.ev_group == "1 strict event")].iloc[0]
ne7 = qa1[(qa1.buffer_days == 7) & (qa1.ev_group == "3 non-event")].iloc[0]
feasible = ev7.n_any_information / ev7.n_admissions >= 0.5
decision("S4 · Feasibility and predictors", [
    ("strict events with any information (7-day margin)", f"{int(ev7.n_any_information)} of {int(ev7.n_admissions)} ({float(ev7.pct_any_information):.1f}%)",
     "the model is feasible" if feasible else "not feasible"),
    ("non-events with any information (7-day margin)", f"{float(ne7.pct_any_information):.1f}%",
     "each block carries an indicator: no record does not mean no condition"),
    ("margins of 1 and 30 days", "almost the same coverage (Q-A1)", "7 days main; 1 and 30 days as sensitivity"),
    ("earlier admissions (Q-A3)", "most were more than a year before (earlier pregnancies)",
     "two indicators: any earlier admission; admission in the 90 days before"),
    ("conditions in earlier admissions (Q-A4)", "frequent ones alone; rare ones in two composites",
     "9 parameters of history (outcome card, section 4)"),
    ("outpatient measurements (Q-A5)", "about 70% with weight and blood pressure, a median of about 2 weeks before",
     "last systolic and diastolic, any pressure of 140/90 or more, last body mass index (280 to 7 days before)"),
    ("outpatient laboratory values (Q-A6)", "hemoglobin in about 70%; creatinine old; A1c ordered selectively",
     "last hemoglobin only; one indicator of prenatal care in this hospital"),
    ("registration variables (Q-A7)", "almost complete", "age, race and ethnicity, insurance, language, marital status; admission type and location excluded (G2)"),
    ("candidate parameters", "27", "reduced set of 17 fixed before any model, for gate G3")])

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 04
notebook_summary([
    ("return used", "R3: antenatal prediction point (information at least 7 days before the admission)"),
    ("population / strict events", lambda: f"{int(l2.n_admissions):,} / {int(l2.n_strict)}"),
    ("gate G1", lambda: "passes" if g1_ok else "DOES NOT PASS"),
    ("strict events with information before the admission (7 days)", lambda: f"{int(ev7.n_any_information)} of {int(ev7.n_admissions)} ({float(ev7.pct_any_information):.1f}%)"),
    ("non-events with information before the admission (7 days)", lambda: f"{float(ne7.pct_any_information):.1f}%"),
    ("feasibility", lambda: "feasible" if feasible else "not feasible"),
    ("candidate parameters", "27 (reduced set: 17)"),
    ("next step", "predictor table, unit and leakage (S5, S6, S7, G2), notebook 05"),
])
write_outputs(OUT_FILE, "Notebook 04 (R3, S2, S3, G1, S4), fetal death MIMIC-IV, antenatal prediction point; aggregate output only")